# XGBoost (weekend gap regression)\n\nGoal: predict **weekend gap %**\n\n`y_gap_pct = (Monday_Open - Friday_Close) / Friday_Close`\n\nUsing gap % is scale-free across tickers and usually easier than predicting raw Monday Open.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parent.parent  # notebooks/models -> repo root
WEEKEND_CSV = REPO_ROOT / "structured_csv_data_files" / "Weekend_dataset.csv"
WEEKEND_CSV

In [ ]:
w = pd.read_csv(WEEKEND_CSV)
w["Date_parsed"] = pd.to_datetime(w["Date"], errors="coerce", utc=True)
w["weekday_num"] = w["Date_parsed"].dt.weekday

for c in ["Open", "Close", "High", "Low", "Volume"]:
    if c in w.columns:
        w[c] = pd.to_numeric(w[c], errors="coerce")

w = w.dropna(subset=["Ticker", "Date_parsed"]).sort_values(["Ticker", "Date_parsed"]).reset_index(drop=True)

w["next_weekday"] = w.groupby("Ticker")["weekday_num"].shift(-1)
w["next_date"] = w.groupby("Ticker")["Date_parsed"].shift(-1)
w["next_open"] = w.groupby("Ticker")["Open"].shift(-1)

pairs = w.loc[(w["weekday_num"] == 4) & (w["next_weekday"] == 0)].copy()
delta_days = (pairs["next_date"] - pairs["Date_parsed"]).dt.days
pairs = pairs.loc[(delta_days >= 3) & (delta_days <= 5)].copy()

pairs["y_gap_pct"] = (pairs["next_open"] - pairs["Close"]) / pairs["Close"]
pairs[["Ticker", "Date_parsed", "next_date", "y_gap_pct"]].head()

In [ ]:
# Features: Friday-only numeric columns
exclude = {
    "Date",
    "Date_parsed",
    "weekday_num",
    "next_weekday",
    "next_date",
    "next_open",
    "y_gap_pct",
}

X_all = pairs[[c for c in pairs.columns if c not in exclude]].select_dtypes(include=["number"]).copy()
y_all = pd.to_numeric(pairs["y_gap_pct"], errors="coerce")

X_all.shape, y_all.describe()

## Train/test split (time-based)

In [ ]:
cutoff = pairs["next_date"].quantile(0.8)
train_idx = pairs["next_date"] <= cutoff
test_idx = ~train_idx

X_train, y_train = X_all.loc[train_idx], y_all.loc[train_idx]
X_test, y_test = X_all.loc[test_idx], y_all.loc[test_idx]

X_train.shape, X_test.shape, cutoff

In [ ]:
# XGBoost model
# If you don't have xgboost installed in your environment:
#   pip install xgboost

from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

from xgboost import XGBRegressor

model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "xgb",
            XGBRegressor(
                n_estimators=800,
                max_depth=5,
                learning_rate=0.05,
                subsample=0.9,
                colsample_bytree=0.9,
                reg_lambda=1.0,
                random_state=42,
                n_jobs=4,
            ),
        ),
    ]
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("RMSE:", mean_squared_error(y_test, pred, squared=False))
print("R2:", r2_score(y_test, pred))

In [ ]:
# Feature importance (gain-based)
booster = model.named_steps["xgb"]
imp = booster.feature_importances_
imp_df = pd.DataFrame({"feature": X_train.columns, "importance": imp}).sort_values("importance", ascending=False)
imp_df.head(30)